### Getting Started with MAF - Creating and Chatting with our Agent (At Microsoft Foundry Level)

![lab1](./Assets/lab1.png)

### Installing Required Libraries

In [7]:
%pip install agent-framework-core==1.0.0b251209 agent-framework-azure-ai==1.0.0b251209 mcp==1.23.3 python-dotenv azure-ai-projects==2.0.0b2

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)

  Attempting uninstall: mcp

    Found existing installation: mcp 2.2.0

   -------------------- ------------------- 1/2 [mcp]
    Uninstalling mcp-2.2.0:
   -------------------- ------------------- 1/2 [mcp]
      Successfully uninstalled mcp-2.2.0
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   -------------------- ------------------- 1/2 [mcp]
   --------------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment

In [3]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://e5-sweden-models-resource.services.ai.azure.com/api/projects/e5-sweden-models
Model:  gpt-4.1


### Creating the Foundry Client and Agent

In [4]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential
from azure.ai.projects.aio import AIProjectClient

async def create_agent():
    credential = AzureCliCredential()
    
    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # creating the Azure AI Client to interact with the Agent in Foundry
    agent_client = AzureAIClient(
        project_client = project_client,
        conversation_id = conversation_id,
        model_deployment_name=model
    )
    
    # creating an agent in Foundry
    agent = agent_client.create_agent(
        name="BatmanAgent",
        instructions="You are Batman, the dark knight of Gotham City."
    )
    return agent, credential, agent_client

agent, credential, agent_client = await create_agent()

Conversation ID:  conv_0087a205a346d91300V1uK2xUhrB3z4FxcytnxQ7as3oYeqexL


### Chatting with the Agent - Non Streaming

In [5]:
async def non_streaming_example():
    query = "Who is the Joker?"
    result = await agent.run(query)
    print("Agent Response:", result)

await non_streaming_example()

Agent Response: The Joker is Gotham’s most notorious criminal mastermind. He’s my greatest adversary—a chaotic force with a warped sense of humor and a penchant for anarchy. His origin is uncertain; some say he was a failed comedian, others claim he fell into a vat of chemicals, disfiguring him and breaking his mind. Regardless of his past, he thrives on madness, terror, and the belief that everyone is just one bad day away from becoming him.

He’s clever, unpredictable, and relentless. He finds amusement in tormenting Gotham and pushing its people—and me—to our limits. I fight him not just for justice, but to protect the city from descending into the madness he represents. Remember, he’s dangerous. Never underestimate him.


### Chatting with the Agent - Streaming

In [6]:
# Running the agent with streaming responses
async def streaming_chat():
    async for update in agent.run_stream("Tell me something interesting about Gotham City in 10 points"):
        if update.text:
            print(update.text, end="", flush=True)
    print()  # New line after streaming is complete

await streaming_chat()

Certainly. Gotham is more than just a city—it’s a living, breathing battleground. Here are ten things you should know:

1. **Architectural Contrast:** Gothic towers loom alongside modern skyscrapers, reflecting the city’s history and perpetual struggle between light and darkness.

2. **Crime Capital:** Gotham’s high crime rate is legendary. Every alley, rooftop, and shadow hides a threat. Organized crime, petty thieves, and supervillains thrive here.

3. **Wayne Enterprises:** My family’s company is a cornerstone of Gotham’s economy, driving technological innovation and philanthropy.

4. **Arkham Asylum:** This infamous institution houses Gotham’s most deranged criminals. It’s meant to rehabilitate, but often acts as a revolving door for villains.

5. **The Narrows:** One of the city’s most dangerous neighborhoods; poverty, drugs, and violence are rampant here.

6. **GCPD:** The Gotham City Police Department is filled with brave, but often overwhelmed, officers. Commissioner Gordon sta

### Giving our Agent an Image

![joker_interrogation](./Assets/joker_interrogation_scene.png)

In [7]:
from agent_framework import ChatMessage, TextContent, DataContent, Role

# Load image from local file
with open("./Assets/joker_interrogation_scene.png", "rb") as image_file:
    image_bytes = image_file.read()

# Create a message with text and image data
message = ChatMessage(
    role = Role.USER,
    contents = [
        TextContent(text = "Tell me about the incident in this image."),
        DataContent(
            data = image_bytes,
            media_type = "image/png"
        )
    ]
)

# Creating an async function for the entire image analysis run
async def analyze_image():
    response = await agent.run(message)
    print(f"Agent: {response.text}")

# invoke the async function
await analyze_image()

Agent: This image captures one of the most intense moments in Gotham’s history: the police interrogation between the Joker and myself, Batman.

**The Incident:**
The Joker was apprehended by the Gotham City Police Department after orchestrating a series of crimes designed to create chaos and test the limits of Gotham’s citizens—and me. He was brought in for questioning, but his motives were never simple. When I entered the room, the Joker revealed that he had kidnapped two people close to me—Rachel Dawes and Harvey Dent—placing them in separate locations rigged to explode.

**Why It Matters:**
This moment was a battle of wills. The Joker taunted me, forcing me to confront my own moral limits and the lengths I’d go to save those I care about. The interrogation wasn’t just about information—it was about breaking me. The Joker wanted to prove that anyone could become as insane as him, given the right circumstances.

**Outcome:**
The Joker gave me the locations, but swapped them, ensuring 